In [ ]:
somegoodmodelexplainationbykachornofc22

https://link.springer.com/article/10.1007/s44163-024-00113-8/figures/2

According to the table, LSTM yielded the best results on the test dataset with a prediction accuracy of 100%. However, when tested in real-time, the accuracy of LSTM dropped to 86%. RNN also achieved 100% accuracy on the test set but showed a lower performance during real-time testing, yielding a result of 64%. On the other hand, Bi-LSTM proved unsuitable for continuous Thai sign language due to the complexity of sequential recognition.

What they did — "The paper compared architectures and found LSTM best on 4 COVID words."
What I did differently & why — "I took LSTM and built the missing applied layer: sentence + voice + deployment, for real communication."
Honest limits + next steps — "My scope is 12 words, one signer; future work is more signers, more words, and my own architecture comparison."

ทำไมตอนมีคำเข้ามาเกี่ยวข้องถึงมี ขอบคุณมา ทั้งๆที่ยังไม่ได้ทำอะไรละ?
Why you kept getting ขอบคุณ over and over
This is the "if there's no hand detected, it still forces Thank You" bug. Here's the exact mechanism — and it ties straight back to the zero-frame story.

The root cause: softmax must pick something
Your model's final layer is softmax over 12 classes — it always outputs a prediction, even for garbage input. When no hand was detected, the input was an all-zero vector ([0,0,0...]). The model can't say "I don't know" — it has to pick the class whose learned pattern is closest to all-zeros.

Why it picked ขอบคุณ specifically
Two reasons stacked:

ขอบคุณ's training data had the most zero-frames. Remember the old extract_keypoints kept no-hand frames. ขอบคุณ is a slower/longer sign, so its clips had more "dead air" (hand resting/leaving frame) → more zero-frames baked into its training samples. So the model literally learned that "lots of zeros = ขอบคุณ."
So when live sent all-zeros (no hand), the closest match was ขอบคุณ. Empty input → "this looks like the zero-heavy ขอบคุณ samples" → predicts ขอบคุณ. Every time your hand dropped or MediaPipe lost it → another ขอบคุณ.
Why it repeated
The old confirmation logic (consecutive counter, no proper vote window or dedup) kept re-confirming the same prediction every cycle. So a hand-down moment didn't give you one stray ขอบคุณ — it spammed ขอบคุณ ขอบคุณ ขอบคุณ... into your buffer.

The three fixes (all of which you now have)
Zero-frame removal (extract fix) → the model no longer associates "empty" with ขอบคุณ, because no training sample is zero-heavy anymore. This kills the root cause.
The none class → now "hand present but not signing" has a real home, so the model isn't forced to pick a word for ambiguous input.
Vote window + dedup in predict_stream.py → even if one stray prediction slips through, it won't get confirmed unless it's consistent, and the same word won't be added twice in a row.

Normal LSTM vs Stacked LSTM
Normal (single) LSTM = one LSTM layer. Reads the sequence once, outputs one summary.
Stacked LSTM = two or more LSTM layers, where the first feeds its output into the second.
Your model has two LSTM layers → that's why it's "Stacked":

LSTM(32, return_sequences=True, input_shape=(30, 126)),  # layer 1
Dropout(0.5),
LSTM(32),                                                # layer 2  ← stacked on top
The key that makes stacking work: return_sequences=True
This is the technical detail that enables the stack:

Layer 1 has return_sequences=True → it outputs a value for all 30 timesteps (a full sequence). It has to, because layer 2 needs a sequence to read.
Layer 2 has no return_sequences (default False) → it outputs only the final summary (one vector), which goes to the Dense layers for the decision.
30 frames → LSTM-1 (returns 30 outputs) → LSTM-2 (returns 1 summary) → Dense → answer
If you removed return_sequences=True from layer 1, the stack would break — layer 1 would output a single vector, and layer 2 (which expects a sequence) would have nothing to chew on.

Why stack — what it buys you
It creates a hierarchy of learning:

Layer 1 learns low-level motion (how fingers/hands move in short spans)
Layer 2 combines those into higher-level patterns (the shape of the whole sign)